In [3]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties

fun ensureFileExists(fileName: String): Boolean {
    val currentDir = System.getProperty("user.dir")
    val file = File(currentDir, fileName)

    if (!file.exists()) {
        file.createNewFile()
        return true
    }
    return false
}

private fun ensureFolderExists(fullPath: String) {
    val currentDir = System.getProperty("user.dir")
    val folder = File(fullPath)

    if (!folder.exists()) {
        folder.mkdirs()
    }
}

fun writeComparingCsv(headers: List<String>, instances: List<String>, revenues: List<List<Int>>, bestInstance: List<String>, fullPath: String) {
    ensureFolderExists(fullPath)

    csvWriter().open("$fullPath/comparison.csv") {
        writeRow(headers)
        instances.forEachIndexed{index, instance ->
            writeRow(listOf(instance) + revenues[index].map { it.toString() } + listOf(bestInstance[index]))
        }
    }

    println("CSV written successfully to $fullPath/comparison.csv")
}

fun <T : Any> writeCsv(data: List<T>, fileName: String, relativePath: String) {
    if (data.isEmpty()) {
        println("No data to write.")
        return
    }

    ensureFolderExists(relativePath)

    val kClass = data.first()::class
    val headers = kClass.declaredMemberProperties.map { it.name }

    csvWriter().open("$relativePath$fileName") {
        writeRow(headers)

        data.forEach { item ->
            val row = kClass.declaredMemberProperties.map { prop ->
                prop.getter.call(item)?.toString()?.replace(",", "\\,") ?: ""
            }
            writeRow(row)
        }
    }

    println("CSV written successfully to $relativePath$fileName")
}


fun compareTwoColumns (col1: DataColumn<*>, col2: DataColumn<*>, name: String): BaseColumn<String> {
    val col = col1.mapIndexed { index, value->
        val refValue = value as Int
        val compValue = col2[index] as Int
        if (refValue <= compValue) {
            (((compValue.toDouble() / refValue.toDouble()) - 1) * 100)
        }else {
            (((refValue.toDouble() / compValue.toDouble()) - 1) * -100)
        }.toInt().toString() + "\\%"
    }
    return col.rename(name)
}

private fun findDifferingSubstring(strings: List<String>): List<String> {
    val split = strings.map { it.split("_") }
    val transposed = split[0].indices.map { i -> split.map { it[i] } }

    val differingIndices = transposed
        .mapIndexedNotNull { index, parts ->
            if (parts.distinct().size > 1) index else null
        }

    return split.map { parts ->
        differingIndices.joinToString("_") { parts[it] }
    }
}
inline fun <reified T> transpose(xs: List<List<T>>): List<List<T>> {
    val cols = xs[0].size
    val rows = xs.size
    return List(cols) { j ->
        List(rows) { i ->
            xs[i][j]
        }
    }
}

fun mergeResults(fullpath: String) {
    val folder = File(fullpath)
    val files =  folder.listFiles()
        ?.filter { it.isFile && it.name != "comparison.csv"}
        ?: emptyList()
    val fileNames = findDifferingSubstring(files.map { it.nameWithoutExtension })

    val results = files.mapIndexed { index, file ->
        fileNames[index] to DataFrame.read(file)
    }
    val instances = results.first().second["name"].map { it.toString() }

    val bestAvgValues = instances.mapIndexed { index, _ ->
        val avgValues = results.map { it.second["revenueAvg"][index] as Int }
        val best = avgValues.maxOrNull()
        val bestIndex = avgValues.indexOf(best)
        results[bestIndex].first
    }
    val avgColumns = results.map {it.first to it.second["revenueAvg"] }

    writeComparingCsv(
        headers = listOf("instance") + fileNames + listOf("best"),
        instances = instances.toList(),
        revenues = transpose(avgColumns.map { it.second.toList().map { value -> value as Int} }),
        bestInstance = bestAvgValues.toList(),
        fullPath = fullpath
    )
}


In [4]:
// static stuff
enum class Context {CLUSTER, BUDGET, CLUSTERKM, CLUSTERALL, ELIMINATION}
enum class Mode {FLAT, RANDOM}
val percentageFraction = 1

val orderedInstances = listOf("eil101", "gil262", "pr299", "lin318", "rd400", "d493", "u574", "u724", "pcb1173", "fl1400", "pr2392").map { if (it == "instance") it else it + "-gen3-50" }

val mode = Mode.RANDOM
val context = Context.CLUSTERALL


val baseline = "fbckmd"
val colsWithoutPercentages = when (context) {
    Context.CLUSTER -> baseline
    Context.BUDGET -> baseline
    Context.CLUSTERKM -> "kmn"
    Context.CLUSTERALL-> baseline
    Context.ELIMINATION -> "OP"
}

val colsToIgnoreWhenCalcuateMax = when (context) {
    Context.CLUSTER -> emptyList()
    Context.BUDGET -> emptyList()
    Context.CLUSTERKM -> emptyList()
    Context.CLUSTERALL -> listOf("kmn", "kmd")
    Context.ELIMINATION -> emptyList()
}

val relativePath = when (context) {
    Context.ELIMINATION -> "/op-solver-strict/results/elimination"
    else -> "/op-solver-strict/results/${context.name.lowercase()}${mode.name.lowercase()}"
}
val header = when (context) {
    Context.CLUSTER -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "best")
    Context.CLUSTERKM -> listOf("instance", "kmn", "kmd", "nckmn", "nckmd", "best")
    Context.CLUSTERALL -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "kmn", "kmd", "best")
    Context.ELIMINATION -> listOf("instance","TSPrfb","TSPrce","OP")
    else -> null
}
data class OneRun(
    val budget: Int,
    val budgetSpentAvg: Double,
    val name: String,
    val revenueAvg: Int,
    val revenueMax: Int,
    val revenueMin: Int,
    val size: Int,
    val successfulAmount: Int,
    val timeAvg: Double,
    val timeMax: Double,
    val timeMin: Double
)

data class ClusterAll(
    val instance: String,
    val rkmn: Int,
    val rkmd: Int,
    val nckmn: Int,
    val nckmd: Int,
    val fbckmn: Int,
    val fbckmd: Int,
    val kmn: Int,
    val kmd: Int,
    val best: String
)

val label = "${context.toString().lowercase()}_${mode.toString().lowercase()}"

val caption = when (context) {
    Context.CLUSTER -> "Comparison of the clustering results for the instances in the ${mode.name.lowercase()} mode."
    Context.BUDGET -> "Comparison of the budget results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERKM -> "Comparison of the k-means impact results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERALL -> "Clustering Algorithm Comparison for ${mode.name.lowercase()} Instances. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at 90\\% of the maximum revenue. The star refers to the best mean revenue (kmn and kmd excluded)."
    Context.ELIMINATION -> "Comparison of the elimination methods for ${mode.name.lowercase()} the instances. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at 90\\% of the maximum revenue. The star refers to the best mean revenue."
}

val title = when (context) {
    Context.CLUSTER -> "clustering"
    Context.BUDGET -> "budget"
    Context.CLUSTERKM -> "k-means impact"
    Context.CLUSTERALL -> "clustering"
    Context.ELIMINATION -> "elimination comparison"
} + " ${mode.name.lowercase()}"

In [5]:
val pathToFolder  = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
.toString() + relativePath

val mainPath = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + relativePath + "/comparison.csv"

if (!File(mainPath).exists()) {
    mergeResults(pathToFolder)
}

var df = DataFrame.readCsv(mainPath).cast<ClusterAll>()
df = df.sortWith (compareBy { row -> orderedInstances.indexOf(row["instance"].toString()) })
    .reorderColumnsBy { colums -> header.indexOf(colums.name()) }
df


java.lang.IndexOutOfBoundsException: Index 0 out of bounds for length 0

In [15]:
val referencePath = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + "/op-solver-strict/results/baseline_${mode.name.lowercase()}.csv"
val baselineDf = DataFrame.readCsv(referencePath).cast<OneRun>()
    .sortWith (compareBy { row -> orderedInstances.indexOf(row["name"].toString()) })

//baselineDf.sortBy { it["revenueAvg"] }.name.values().map { it.split("-").first() }.map { "\"$it\"" }

baselineDf

budget,budgetSpentAvg,name,revenueAvg,revenueMax,revenueMin,size,successfulAmount,timeAvg,timeMax,timeMin
315,305.730000,eil101,3272,3340,3197,101,5,1.160000,1.460000,0.590000
1189,1171.664000,gil262,7427,7590,7185,262,5,20.110000,43.120000,6.920000
24096,23646.498000,pr299,8650,9075,8403,299,5,7.080000,10.750000,5.050000
21015,20773.220000,lin318,9975,10136,9537,318,5,24.740000,76.000000,9.800000
7641,7536.742000,rd400,11866,12004,11785,400,5,17.220000,21.650000,13.450000
17501,17200.418000,d493,16503,16795,16334,493,5,23.250000,32.980000,13.180000
18453,18173.558000,u574,16704,16971,16210,574,5,25.660000,32.400000,18.510000
20955,20174.364000,u724,17349,21030,3905,724,5,53.230000,113.550000,0.680000
28446,27951.962000,pcb1173,31503,32039,31035,1173,5,36.960000,61.290000,25.400000
10064,9832.797500,fl1400,49362,49921,48768,1400,4,105.190000,133.240000,75.440000


In [16]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

In [17]:
df = df.remove("best")
df.update("instance").with { it.toString().split("-")[0] }
val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{1.7cm}|" }
val header = df.columnNames().joinToString(separator = " & ")

df

instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd
eil101,3223,3232,3286,3224,3279,3289,3297,3285
gil262,7159,7129,7298,7487,7144,7256,7248,7319
pr299,8492,8835,8394,8685,8441,8606,8591,8638
lin318,9602,9676,9711,9624,9879,9859,10073,10060
rd400,11433,11511,11846,11952,11820,11703,11622,11654
d493,15786,15681,14989,15369,15836,15918,16163,16122
u574,16206,16264,16751,16672,16687,16678,17118,16744
u724,20463,20320,20542,20639,20718,21097,21163,21111
pcb1173,30797,30750,31694,31212,31379,32198,31983,32142
fl1400,51387,47123,49028,48345,48938,47028,51633,52999


In [18]:
val bestValues = df.convert { all()}.perRowCol { row, col ->
    if (col[row] is String || colsToIgnoreWhenCalcuateMax.contains(col.name())) {
        0
    } else {
        col[row] as Int
    }
}.map{ row ->
    row.rowMaxOf<Int>()
}

bestValues


[3289, 7487, 8835, 9879, 11952, 15918, 16751, 21097, 32198, 51387, 66370]

In [24]:


val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax as Int, baselineDf["revenueAvg"][index] as Int) }

val rowMinValues = df.map { row ->
    row.rowMinOf<Int>()
}

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Int>()
    (maxEntry - minEntry).toDouble() / maxEntry.toDouble()
}
val maxRevenueDif = revenueDif.max()
val gradient = 0.1 //max(maxRevenueDif,0.0)

fun getSaturation (gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100-((maxValue - value) / (maxValue * gradient) * 100)),0.0),100.0).toInt().toString()
}

rowMaxValues


[3289, 7487, 8835, 9975, 11952, 16503, 16751, 21097, 32198, 51387, 66370]

In [20]:
import java.util.Locale

fun calculatePercentage(refValue: Int, compValue: Int): Double{ return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())}
fun formatePercentage(value: Double): String { return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%" }

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = df.get(colsWithoutPercentages)[row] as Int
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = df.get(colsWithoutPercentages)[row] as Int
        calculatePercentage(refValue, col[row] as Int)
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it)
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -1.3\%, -1.7\%, -0.9\%, -0.7\%, -0.4\%, -, +1.3\%, +1.6\%]

In [21]:
val stringdf = df.convert { all() }.perRowCol { row, col  ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = col[row] as Int
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())){
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        }else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = col[row] as Int
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        }else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd
eil101,\cellcolor{cyan!79} 3223{\tiny-2.0\%},\cellcolor{cyan!82} 3232{\tiny-1.7\%},\cellcolor{cyan!99} 3286{\tiny-0.1\%},\cellcolor{cyan!80} 3224{\tiny-2.0\%},\cellcolor{cyan!96} 3279{\tiny-0.3\%},\cellcolor{cyan!100} \textbf{3289*},\cellcolor{cyan!100} 3297{\tiny+0.2\%},\cellcolor{cyan!98} 3285{\tiny-0.1\%}
gil262,\cellcolor{cyan!56} 7159{\tiny-1.3\%},\cellcolor{cyan!52} 7129{\tiny-1.8\%},\cellcolor{cyan!74} 7298{\tiny+0.6\%},\cellcolor{cyan!100} \textbf{7487*}{\...,\cellcolor{cyan!54} 7144{\tiny-1.5\%},\cellcolor{cyan!69} 7256,\cellcolor{cyan!68} 7248{\tiny-0.1\%},\cellcolor{cyan!77} 7319{\tiny+0.9\%}
pr299,\cellcolor{cyan!61} 8492{\tiny-1.3\%},\cellcolor{cyan!100} \textbf{8835*}{\...,\cellcolor{cyan!50} 8394{\tiny-2.5\%},\cellcolor{cyan!83} 8685{\tiny+0.9\%},\cellcolor{cyan!55} 8441{\tiny-1.9\%},\cellcolor{cyan!74} 8606,\cellcolor{cyan!72} 8591{\tiny-0.2\%},\cellcolor{cyan!77} 8638{\tiny+0.4\%}
lin318,\cellcolor{cyan!62} 9602{\tiny-2.6\%},\cellcolor{cyan!70} 9676{\tiny-1.9\%},\cellcolor{cyan!73} 9711{\tiny-1.5\%},\cellcolor{cyan!64} 9624{\tiny-2.4\%},\cellcolor{cyan!90} \textbf{9879*}{\t...,\cellcolor{cyan!88} 9859,\cellcolor{cyan!100} 10073{\tiny+2.2\%},\cellcolor{cyan!100} 10060{\tiny+2.0\%}
rd400,\cellcolor{cyan!56} 11433{\tiny-2.3\%},\cellcolor{cyan!63} 11511{\tiny-1.6\%},\cellcolor{cyan!91} 11846{\tiny+1.2\%},\cellcolor{cyan!100} \textbf{11952*}{...,\cellcolor{cyan!88} 11820{\tiny+1.0\%},\cellcolor{cyan!79} 11703,\cellcolor{cyan!72} 11622{\tiny-0.7\%},\cellcolor{cyan!75} 11654{\tiny-0.4\%}
d493,\cellcolor{cyan!56} 15786{\tiny-0.8\%},\cellcolor{cyan!50} 15681{\tiny-1.5\%},\cellcolor{cyan!8} 14989{\tiny-5.8\%},\cellcolor{cyan!31} 15369{\tiny-3.4\%},\cellcolor{cyan!59} 15836{\tiny-0.5\%},\cellcolor{cyan!64} \textbf{15918*},\cellcolor{cyan!79} 16163{\tiny+1.5\%},\cellcolor{cyan!76} 16122{\tiny+1.3\%}
u574,\cellcolor{cyan!67} 16206{\tiny-2.8\%},\cellcolor{cyan!70} 16264{\tiny-2.5\%},\cellcolor{cyan!100} \textbf{16751*}{...,\cellcolor{cyan!95} 16672{\tiny-0.0\%},\cellcolor{cyan!96} 16687{\tiny+0.1\%},\cellcolor{cyan!95} 16678,\cellcolor{cyan!100} 17118{\tiny+2.6\%},\cellcolor{cyan!99} 16744{\tiny+0.4\%}
u724,\cellcolor{cyan!69} 20463{\tiny-3.0\%},\cellcolor{cyan!63} 20320{\tiny-3.7\%},\cellcolor{cyan!73} 20542{\tiny-2.6\%},\cellcolor{cyan!78} 20639{\tiny-2.2\%},\cellcolor{cyan!82} 20718{\tiny-1.8\%},\cellcolor{cyan!100} \textbf{21097*},\cellcolor{cyan!100} 21163{\tiny+0.3\%},\cellcolor{cyan!100} 21111{\tiny+0.1\%}
pcb1173,\cellcolor{cyan!56} 30797{\tiny-4.4\%},\cellcolor{cyan!55} 30750{\tiny-4.5\%},\cellcolor{cyan!84} 31694{\tiny-1.6\%},\cellcolor{cyan!69} 31212{\tiny-3.1\%},\cellcolor{cyan!74} 31379{\tiny-2.5\%},\cellcolor{cyan!100} \textbf{32198*},\cellcolor{cyan!93} 31983{\tiny-0.7\%},\cellcolor{cyan!98} 32142{\tiny-0.2\%}
fl1400,\cellcolor{cyan!100} \textbf{51387*}{...,\cellcolor{cyan!17} 47123{\tiny+0.2\%},\cellcolor{cyan!54} 49028{\tiny+4.3\%},\cellcolor{cyan!40} 48345{\tiny+2.8\%},\cellcolor{cyan!52} 48938{\tiny+4.1\%},\cellcolor{cyan!15} 47028,\cellcolor{cyan!100} 51633{\tiny+9.8\%},\cellcolor{cyan!100} 52999{\tiny+12.7\%}


In [22]:
val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ")  {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ")  {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|  }
                \hline
                \multicolumn{9}{|c|}{clustering random} \\
                \hline
                    instance & rkmn & rkmd & nckmn & nckmd & fbckmn & fbckmd & kmn & kmd \\
                \hline
                    eil101 & \cellcolor{cyan!79} 3223{\tiny-2.0\%} & \cellcolor{cyan!82} 3232{\tiny-1.7\%} & \cellcolor{cyan!99} 3286{\tiny-0.1\%} & \cellcolor{cyan!80} 3224{\tiny-2.0\%} & \cellcolor{cyan!96} 3279{\tiny-0.3\%} & \cellcolor{cyan!100} \textbf{3289*} & \cellcolor{cyan!100} 3297{\tiny+0.2\%} & \cellcolor{cyan!98} 3285{\tiny-0.1\%} \\ 
gil262 & \cellcolor{cyan!56} 7159{\tiny-1.3\%} & \cellcolor{cyan!52} 7129{\tiny-1.8\%} & \cellcolor{cyan!74} 7298{\tiny+0.6\%} & \cellcolor{cyan!100} \textbf{7487*}{\tiny+3.2\%} & \cellcolor{cyan!54} 7144{\tiny-1.5\%} & \cellcolor{cyan!69}